# imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
import sys
sys.path.append("code")
import final_functions as ff

In [ ]:
import importlib
importlib.reload(ff)
pass

# functions

In [ ]:
def get_p_value(real_value, null_values, direction):
    """direction should be "greater" or "less" depending on whether the real_value is intended to be greater or less than the null_values when it is significant
    returns p-value and num_extreme"""

    if direction == "greater":
        num_extreme = sum(1 for null_value in null_values if null_value >= real_value)
    elif direction == "less":
        num_extreme = sum(1 for null_value in null_values if null_value <= real_value)
    else:
        raise Exception("direction must be 'greater' or 'less'")
    
    return num_extreme / len(null_values), num_extreme

def get_effect_size(real_value, null_values, direction):
    """direction should be "greater" or "less" depending on whether the real_value is intended to be greater or less than the null_values when it is significant"""

    mean_null = np.mean(null_values)
    std_null = np.std(null_values)
        
    effect = (real_value - mean_null) / std_null
    
    return effect if direction == "greater" else -1 * effect

def plot_true_vs_null_values(true_value, null_values, plot_type, num_bins=None, x_axis_name=None, y_axis_name=None, title=None, font_size=None):
    """plot_type must be "bar" or "histogram"
    optional param num_bins for if plot_type is histogram (default 10)"""

    plt.figure(figsize=(10, 8), dpi=200)

    if (plot_type == "histogram"):
        plt.hist(null_values, bins=num_bins if num_bins is not None else 10, color='blue', edgecolor='white')
        plt.axvline(true_value, color='red', linestyle='--', linewidth=2)

    elif (plot_type == "bar"):

        rounded_null_values = [round(null_value, 10) for null_value in null_values]
        counts = Counter(rounded_null_values)
        xs = sorted(counts.keys())
        ys = [counts[x] for x in xs]

        if len(xs) > 1:
            min_diff = min(xs[i+1] - xs[i] for i in range(len(xs) - 1))
            bar_width = min_diff * 0.8 
        else:
            bar_width = 0.5
        
        plt.bar(xs, ys, color='blue', edgecolor='white', width=bar_width)
        plt.axvline(true_value, color='red', linestyle='--', linewidth=2)
    
        for x, y in zip(xs, ys):
            plt.text(x, y + max(ys)*0.015, str(y), ha='center', va='center', fontsize=font_size)

    else: raise Exception("plot_type must be 'bar' or 'histogram'")

    plt.tick_params(axis='both', labelsize=font_size)

    plt.title(title, fontsize=font_size)
    plt.xlabel(x_axis_name, fontsize=font_size)
    plt.ylabel(y_axis_name, fontsize=font_size)
    
    plt.tight_layout()
    plt.show()

# AML

In [ ]:
# for aml, 3_stages_0_min_alt_freq_None_mapp

In [ ]:
columns = ["seed", "num_null_datasets", "num_null_runs", "total_time_took", "total_time_took_problems", "total_time_took_other",
           "num_stages", "num_pathways", "num_dependencies", "max_alt_per_pathway", "min_alt_freq", "num_alt", 
           "true_correct", "null_correct_all", "true_time_took_problem", "null_time_took_problem_avg", "true_time_took_other", "null_time_took_other_avg", 
           "true_avg_alterations_per_pathway", "null_avg_alterations_per_pathway_avg", 
           "true_strict_score", "null_strict_score_avg", "true_strict_p1_value", "null_strict_p1_value_avg", "true_strict_effect_size", "null_strict_effect_size_avg", "strict_p2_value_of_effect1_size", 
           "true_partial_score", "null_partial_score_avg", "true_partial_p1_value", "null_partial_p1_value_avg", "true_partial_effect_size", "null_partial_effect_size_avg", "partial_p2_value_of_effect1_size", 
           "true_stage_score", "null_stage_score_avg", "true_stage_score_p1_value", "null_stage_score_p1_value_avg", "true_stage_score_effect_size", "null_stage_score_effect_size_avg", "stage_score_p2_value_of_effect1_size"]

results_file_path = os.path.join(ff.get_file_path("project_root"), "final", "results", "main_results_significance_testing", "AML_3_0_None")

patient_trees = ff.get_patient_trees_lists("AML")
min_alteration_frequency = 0
num_stages = 3
num_pathways = num_stages
num_dependencies = num_stages - 1
max_alterations_per_pathway = None
num_null_datasets = 1000
num_null_runs = 250
seed = 1

AML_result = ff.run_problem_1_1_full_significance_testing(patient_trees, min_alteration_frequency, num_stages, num_pathways, num_dependencies, max_alterations_per_pathway, num_null_datasets, num_null_runs, seed, results_file_path, TRACERx=False)
result_copy = AML_result.copy()

result_copy[13] = all(AML_result[13])
result_copy[15] = np.mean(AML_result[15])
result_copy[17] = np.mean(AML_result[17])
result_copy[19] = np.mean(AML_result[19])
result_copy[21] = np.mean(AML_result[21])
result_copy[23] = np.mean(AML_result[23])
result_copy[25] = np.mean(AML_result[25])
result_copy[28] = np.mean(AML_result[28])
result_copy[30] = np.mean(AML_result[30])
result_copy[32] = np.mean(AML_result[32])
result_copy[35] = np.mean(AML_result[35])
result_copy[37] = np.mean(AML_result[37])
result_copy[39] = np.mean(AML_result[39])

results = [result_copy]
df = pd.DataFrame(results, columns=columns)

df.to_csv(os.path.join(results_file_path, "summary.csv"), index=False)

## plotting

In [ ]:
num_null_runs = AML_result[2]

true_strict_score = AML_result[20]
null_strict_score_list = AML_result[21]
true_strict_p1_value = AML_result[22]
null_strict_p1_value_list = AML_result[23]
true_strict_effect1_size = AML_result[24]
null_strict_effect1_size_list = AML_result[25]
strict_p2_value_of_effect1_size = AML_result[26]

true_partial_score = AML_result[27]
null_partial_score_list = AML_result[28]
true_partial_p1_value = AML_result[29]
null_partial_p1_value_list = AML_result[30]
true_partial_effect1_size = AML_result[31]
null_partial_effect1_size_list = AML_result[32]
partial_p2_value_of_effect1_size = AML_result[33]

true_stage_score = AML_result[34]
null_stage_score_list = AML_result[35]
true_stage_score_p1_value = AML_result[36]
null_stage_score_p1_value_list = AML_result[37]
true_stage_score_effect1_size = AML_result[38]
null_stage_score_effect1_size_list = AML_result[39]
stage_score_p2_value_of_effect1_size = AML_result[40]

In [ ]:
direction = "greater"
true_value = true_strict_effect1_size
null_values = null_strict_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Strict Effect1-Size", y_axis_name="Frequency", title="Distribution of Strict Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of strict score = {effect_size}")

### plotting for strict score

In [ ]:
direction = "greater"
true_value = true_strict_score
null_values = null_strict_score_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Strict Score", y_axis_name="Frequency", title="Distribution of Strict Scores")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of strict score = {effect_size}")

In [ ]:
direction = "less"
true_value = true_strict_p1_value
null_values = null_strict_p1_value_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Strict P1-Value", y_axis_name="Frequency", title="Distribution of Strict P1-Values")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of p1-value of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of p1-value of strict score = {effect_size}")

In [ ]:
direction = "greater"
true_value = true_strict_effect1_size
null_values = null_strict_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Strict Effect1-Size", y_axis_name="Frequency", title="Distribution of Strict Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of strict score = {effect_size}")

### plotting for partial score

In [ ]:
direction = "greater"
true_value = true_partial_score
null_values = null_partial_score_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Partial Score", y_axis_name="Frequency", title="Distribution of Partial Scores")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of partial score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of partial score = {effect_size}")

In [ ]:
direction = "less"
true_value = true_partial_p1_value
null_values = null_partial_p1_value_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Partial P1-Value", y_axis_name="Frequency", title="Distribution of Partial P1-Values")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of p1-value of partial score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of p1-value of partial score = {effect_size}")

In [ ]:
direction = "greater"
true_value = true_partial_effect1_size
null_values = null_partial_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Partial Effect1-Size", y_axis_name="Frequency", title="Distribution of Partial Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of partial score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of partial score = {effect_size}")

### plotting for stage score

In [ ]:
direction = "greater"
true_value = true_stage_score
null_values = null_stage_score_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Stage Score", y_axis_name="Frequency", title="Distribution of Stage Scores")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of stage score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of stage score = {effect_size}")

In [ ]:
direction = "less"
true_value = true_stage_score_p1_value
null_values = null_stage_score_p1_value_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Stage Score P1-Value", y_axis_name="Frequency", title="Distribution of Stage Score P1-Values")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of p1-value of stage score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of p1-value of stage score = {effect_size}")

In [ ]:
direction = "greater"
true_value = true_stage_score_effect1_size
null_values = null_stage_score_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Stage Score Effect1-Size", y_axis_name="Frequency", title="Distribution of Stage Score Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of stage score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of stage score = {effect_size}")

# TRACERx

In [ ]:
# for tracerx, 4_stages_8_min_alt_freq_5_mapp

In [ ]:
columns = ["seed", "num_null_datasets", "num_null_runs", "total_time_took", "total_time_took_problems", "total_time_took_other",
           "num_stages", "num_pathways", "num_dependencies", "max_alt_per_pathway", "min_alt_freq", "num_alt", 
           "true_correct", "null_correct_all", "true_time_took_problem", "null_time_took_problem_avg", "true_time_took_other", "null_time_took_other_avg", 
           "true_avg_alterations_per_pathway", "null_avg_alterations_per_pathway_avg", 
           "true_num_strictly_consistent_patients", "null_num_strictly_consistent_patients_avg", "true_strict_p1_value", "null_strict_p1_value_avg", "true_strict_effect_size", "null_strict_effect_size_avg", "strict_p2_value", 
           "true_num_partially_consistent_patients", "null_num_partially_consistent_patients_avg", "true_partial_p1_value", "null_partial_p1_value_avg", "true_partial_effect_size", "null_partial_effect_size_avg", "partial_p2_value", 
           "true_stage_score", "null_stage_score_avg", "true_stage_score_p1_value", "null_stage_score_p1_value_avg", "true_stage_score_effect_size", "null_stage_score_effect_size_avg", "stage_score_p2_value"]

results_file_path = os.path.join(ff.get_file_path("project_root"), "final", "results", "main_results_significance_testing", "TRACERx_4_8_5")

patient_trees = ff.get_patient_trees_lists("TRACERx")
min_alteration_frequency = 8
num_stages = 4
num_pathways = num_stages
num_dependencies = num_stages - 1
max_alterations_per_pathway = 5
num_null_datasets = 1000
num_null_runs = 250
seed = 1

TRACERx_result = ff.run_problem_1_1_full_significance_testing(patient_trees, min_alteration_frequency, num_stages, num_pathways, num_dependencies, max_alterations_per_pathway, num_null_datasets, num_null_runs, seed, results_file_path, TRACERx=True)
result_copy = TRACERx_result.copy()

result_copy[13] = all(TRACERx_result[13])
result_copy[15] = np.mean(TRACERx_result[15])
result_copy[17] = np.mean(TRACERx_result[17])
result_copy[19] = np.mean(TRACERx_result[19])
result_copy[21] = np.mean(TRACERx_result[21])
result_copy[23] = np.mean(TRACERx_result[23])
result_copy[25] = np.mean(TRACERx_result[25])
result_copy[28] = np.mean(TRACERx_result[28])
result_copy[30] = np.mean(TRACERx_result[30])
result_copy[32] = np.mean(TRACERx_result[32])
result_copy[35] = np.mean(TRACERx_result[35])
result_copy[37] = np.mean(TRACERx_result[37])
result_copy[39] = np.mean(TRACERx_result[39])

results = [result_copy]
df = pd.DataFrame(results, columns=columns)

df.to_csv(os.path.join(results_file_path, "summary.csv"), index=False)

## plotting

In [ ]:
num_null_runs = TRACERx_result[2]

true_strict_score = TRACERx_result[20]
null_strict_score_list = TRACERx_result[21]
true_strict_p1_value = TRACERx_result[22]
null_strict_p1_value_list = TRACERx_result[23]
true_strict_effect1_size = TRACERx_result[24]
null_strict_effect1_size_list = TRACERx_result[25]
strict_p2_value_of_effect1_size = TRACERx_result[26]

true_partial_score = TRACERx_result[27]
null_partial_score_list = TRACERx_result[28]
true_partial_p1_value = TRACERx_result[29]
null_partial_p1_value_list = TRACERx_result[30]
true_partial_effect1_size = TRACERx_result[31]
null_partial_effect1_size_list = TRACERx_result[32]
partial_p2_value_of_effect1_size = TRACERx_result[33]

true_stage_score = TRACERx_result[34]
null_stage_score_list = TRACERx_result[35]
true_stage_score_p1_value = TRACERx_result[36]
null_stage_score_p1_value_list = TRACERx_result[37]
true_stage_score_effect1_size = TRACERx_result[38]
null_stage_score_effect1_size_list = TRACERx_result[39]
stage_score_p2_value_of_effect1_size = TRACERx_result[40]

### plotting for strict score

In [ ]:
direction = "greater"
true_value = true_strict_score
null_values = null_strict_score_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Strict Score", y_axis_name="Frequency", title="Distribution of Strict Scores")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of strict score = {effect_size}")

In [ ]:
direction = "less"
true_value = true_strict_p1_value
null_values = null_strict_p1_value_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Strict P1-Value", y_axis_name="Frequency", title="Distribution of Strict P1-Values")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of p1-value of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of p1-value of strict score = {effect_size}")

In [ ]:
direction = "greater"
true_value = true_strict_effect1_size
null_values = null_strict_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Strict Effect1-Size", y_axis_name="Frequency", title="Distribution of Strict Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of strict score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of strict score = {effect_size}")

### plotting for partial score

In [ ]:
direction = "greater"
true_value = true_partial_score
null_values = null_partial_score_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Partial Score", y_axis_name="Frequency", title="Distribution of Partial Scores")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of partial score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of partial score = {effect_size}")

In [ ]:
direction = "less"
true_value = true_partial_p1_value
null_values = null_partial_p1_value_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Partial P1-Value", y_axis_name="Frequency", title="Distribution of Partial P1-Values")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of p1-value of partial score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of p1-value of partial score = {effect_size}")

In [ ]:
direction = "greater"
true_value = true_partial_effect1_size
null_values = null_partial_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Partial Effect1-Size", y_axis_name="Frequency", title="Distribution of Partial Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of partial score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of partial score = {effect_size}")

### plotting for stage score

In [ ]:
direction = "greater"
true_value = true_stage_score
null_values = null_stage_score_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="bar", x_axis_name="Stage Score", y_axis_name="Frequency", title="Distribution of Stage Scores")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of stage score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of stage score = {effect_size}")

In [ ]:
direction = "less"
true_value = true_stage_score_p1_value
null_values = null_stage_score_p1_value_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Stage Score P1-Value", y_axis_name="Frequency", title="Distribution of Stage Score P1-Values")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of p1-value of stage score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of p1-value of stage score = {effect_size}")

In [ ]:
direction = "greater"
true_value = true_stage_score_effect1_size
null_values = null_stage_score_effect1_size_list

plot_true_vs_null_values(true_value=true_value, null_values=null_values, plot_type="histogram", x_axis_name="Stage Score Effect1-Size", y_axis_name="Frequency", title="Distribution of Stage Score Effect1-Sizes")

p_value, num_extreme = get_p_value(true_value, null_values, direction)
effect_size = get_effect_size(true_value, null_values, direction)

print(f"p2-value of effect1-size of stage score = {p_value} ({num_extreme} / {num_null_runs})")
print(f"effect2-size of effect1-size of stage score = {effect_size}")